In [ ]:
# Cell 1: Dependencies
#!pip install google-generativeai datasets pandas pyarrow huggingface_hub langid

# Cell 2: Imports
import pandas as pd
import numpy as np
from datasets import Dataset, load_dataset, DatasetDict
import os
import json
import time
import google.generativeai as genai
from datetime import datetime
import re
import langid
from collections import Counter
import hashlib
from huggingface_hub import login, upload_file, hf_hub_download

In [ ]:
# Cell 3: Enhanced Contract Corpus Generation with Analysis
DOMAIN_CONFIG = {
    'domain': 'Legal Contract & Agreement Corpus',
    'target_language': 'en',  # or 'id' for Indonesian
    'generation_mode': 'full_contract',
    'include_fairness_analysis': True,
}

# Contract taxonomy for permutation/combination
CONTRACT_TAXONOMY = {
    'agreement_types': [
        "Non-Disclosure Agreement (NDA)",
        "Master Services Agreement (MSA)",
        "Statement of Work (SOW)",
        "Service Level Agreement (SLA)",
        "Software License Agreement (EULA)",
        "Shareholders Agreement",
        "Employment Agreement",
        "Partnership Agreement",
        "Joint Venture Agreement",
        "Commercial Lease Agreement",
        "Supply Agreement",
        "Distribution Agreement",
        "Franchise Agreement",
        "Consulting Agreement",
        "Software Development Agreement",
    ],
    
    'industries': [
        "Information Technology (IT) & Software Development",
        "Artificial Intelligence & Machine Learning",
        "Banking, Fintech & Financial Services",
        "Healthcare, Pharmaceuticals & Biotechnology",
        "Construction & Real Estate Development",
        "Energy, Oil & Gas, & Utilities",
        "Manufacturing & Heavy Industry",
        "Government & Public Sector",
        "Retail & E-commerce",
        "Telecommunications",
        "Transportation & Logistics",
    ],
    
    'services': [
        "Software Development & Integration",
        "SaaS / Cloud Services (IaaS, PaaS)",
        "AI Model Development & API Integration",
        "Strategic Management Consulting",
        "Legal Advisory & Compliance Review",
        "Facility Management & Maintenance",
        "Data Analytics & Business Intelligence",
        "Cybersecurity Services",
        "Marketing & Digital Advertising",
        "Supply Chain Management",
    ],
    
    'products': [
        "Cloud-Based SaaS Platform",
        "Enterprise Hardware (Servers, Storage)",
        "AI Model License or Dataset Access",
        "Pharmaceutical Compounds & Medical Devices",
        "Construction Materials (cement, steel, glass)",
        "Consumer Electronics",
        "Industrial Equipment",
        "Software Licenses",
    ],
    
    'relationships': [
        "B2B (Business-to-Business)",
        "B2G (Business-to-Government)",
        "Employer-Employee",
        "Partner-Partner (Joint Venture)",
        "Licensor-Licensee (IP Transfer)",
        "Vendor-Client",
        "Franchisor-Franchisee",
    ],
    
    'critical_clauses': [
        "Confidentiality & Non-Disclosure",
        "Intellectual Property Ownership & Licensing",
        "Limitation of Liability & Damages Cap",
        "Indemnification & Defense Obligations",
        "Data Protection & Privacy Compliance (GDPR, HIPAA, PDPA)",
        "Dispute Resolution (Arbitration / Mediation)",
        "Payment Terms, Invoicing & Tax Obligations",
        "Termination for Convenience / Cause",
        "Force Majeure",
        "Warranties and Representations",
        "Non-Compete & Non-Solicitation",
        "Audit Rights",
    ],
    
    'jurisdictions': [
        "US Federal & State Law",
        "European Union (EU) Law",
        "Singapore & ASEAN Regional Law",
        "Indonesia (Civil Law System)",
        "International (UNCITRAL / CISG Framework)",
        "United Kingdom",
        "Australia",
        "Canada",
    ],
    
    'metadata': {
        'risk_profile': ["Low", "Medium", "High", "Critical"],
        'duration': ["<1 year", "1-3 years", "3-5 years", "5+ years", "Perpetual"],
        'contract_value': ["<50K", "50K-500K", "500K-5M", "5M-50M", ">50M USD"],
    }
}

# Optimized single-shot generation prompt
CONTRACT_GENERATION_PROMPT = """Generate a complete, realistic {agreement_type} between two commercial parties.

BUSINESS CONTEXT:
- Industry Sector: {industry}
- Subject Matter: {service_product}
- Party Relationship: {relationship}
- Contract Duration: {duration}
- Estimated Value: {value}
- Risk Level: {risk_profile}
- Governing Law: {jurisdiction}

REQUIRED ELEMENTS (naturally integrated, not as template):
Include these key provisions where relevant: {clauses}

INSTRUCTIONS:
1. Create realistic party names (companies/entities appropriate for {industry})
2. Use authentic legal language for {jurisdiction}
3. Structure naturally - sections should flow logically based on agreement type
4. Include realistic specifics: dates, amounts, deliverables, milestones, KPIs
5. Vary section depth - critical provisions should be detailed, standard ones concise
6. Add appropriate schedules/exhibits if the agreement type typically includes them
7. Write as a practicing attorney would - NOT from a template

The contract should feel like a real negotiated document with:
- Specific commercial terms (not generic placeholders)
- Industry-appropriate language and standards
- Natural variation in clause structure and detail
- Realistic party names and addresses
- Actual numbers, percentages, deadlines

Generate the complete agreement now. Be thorough where complexity demands it, concise where standard.

AGREEMENT DOCUMENT:"""

# Lightweight fairness analysis prompt (only if needed)
FAIRNESS_ANALYSIS_PROMPT = """Analyze the power dynamics and fairness of this contract excerpt:

{contract_preview}

Context: {agreement_type} in {industry} sector
Parties: {relationship}

Provide a concise analysis (400-500 words):

1. BALANCE ASSESSMENT:
   - Overall fairness score (1-10)
   - Which party has stronger position
   - Key power imbalances

2. RISK ALLOCATION:
   - Financial risk distribution
   - Performance obligations balance
   - Exit/termination rights

3. NEGOTIATION POINTS:
   - Provisions favoring Party A
   - Provisions favoring Party B
   - Balanced terms

Keep analysis focused and actionable.

ANALYSIS:"""

# System prompt
SYSTEM_PROMPT = """You are a senior commercial attorney with 20+ years of experience drafting complex agreements across multiple industries and jurisdictions. 

You draft realistic, enforceable contracts that reflect actual business negotiations - not generic templates. Your contracts include:
- Specific commercial terms with real numbers
- Industry-appropriate provisions and standards
- Natural language variations (not formulaic)
- Realistic party dynamics and negotiated positions
- Proper legal structure for the jurisdiction

You adapt your drafting style to the agreement type, industry, and risk profile."""

In [ ]:
# Cell 4: Main Configuration
CONFIG = {
    #'gemini_api_key': '',
    'gemini_api_key': '',
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    
    'huggingface_token': '',
    'output_repository': 'Azzindani/Legal_Contract_Syn',
    
    # Processing settings
    'batch_size': 5,
    'model_name': 'gemini-2.5-flash',  # Use flash-exp for better output/cost ratio
    'temperature': 0.85,  # Higher for more variation
    
    # Token optimization
    'max_output_tokens': 50000,  # Allow full contracts
    'enable_fairness_analysis': False,  # Set True only if you need detailed analysis (costs extra tokens)
    
    # Quality settings
    'min_contract_length': 5000,  # Characters minimum
    'target_language_confidence': 0.5,
    
    # Processing control
    'skip_processed_rows': True,
    'progress_file': 'synthesis_progress.json',

    'enable_fairness_analysis': True,  # Adds ~2000 tokens per contract
    
    # Safety settings (allow legal terminology)
    'safety_settings': {
        'HARM_CATEGORY_HARASSMENT': 'BLOCK_NONE',
        'HARM_CATEGORY_HATE_SPEECH': 'BLOCK_NONE',
        'HARM_CATEGORY_SEXUALLY_EXPLICIT': 'BLOCK_NONE',
        'HARM_CATEGORY_DANGEROUS_CONTENT': 'BLOCK_ONLY_HIGH',
    }
}

In [ ]:
# Cell 5: Authentication
genai.configure(api_key=CONFIG['gemini_api_key'])
login(token=CONFIG['huggingface_token'])

In [ ]:
# Cell 6: Text Validator for Single-Shot Processing
class TextValidator:
    def __init__(self):
        self.max_length = DOMAIN_CONFIG['max_text_length']
        self.min_length = DOMAIN_CONFIG['min_text_length']
    
    def validate_and_prepare_text(self, text):
        """Validate and prepare text for single-shot processing"""
        if not text or not isinstance(text, str):
            return None
        
        text = text.strip()
        
        if len(text) < self.min_length:
            return None
        
        # Truncate if too long (preserve beginning and end)
        if len(text) > self.max_length:
            half_max = self.max_length // 2 - 100  # Leave room for separator
            text = text[:half_max] + "\n\n[...TRUNCATED...]\n\n" + text[-half_max:]
        
        return text

In [ ]:
# Cell 7: FIXED Language Detection
class LanguageScorer:
    def __init__(self, target_language='en'):
        self.target_language = target_language
    
    def detect_language(self, text):
        """Detect language - FIX for langid's negative confidence"""
        if not text or len(text.strip()) < 10:
            return 'unknown', 0.0
        
        try:
            lang, raw_confidence = langid.classify(text.strip())
            
            # CRITICAL FIX: langid returns NEGATIVE log probability
            # Convert to 0-1 scale using exponential
            # More negative = less confident, closer to 0 = more confident
            import math
            
            # Normalize: convert negative log prob to probability
            # Typical range is -inf to 0, most values between -5 and -0.5
            if raw_confidence <= 0:
                # More confident (closer to 0) = higher score
                normalized_confidence = math.exp(raw_confidence)
                # This gives us 0.0 to 1.0 where 1.0 is most confident
            else:
                # Shouldn't happen, but safety check
                normalized_confidence = 1.0
            
            # Clamp to valid range
            normalized_confidence = max(0.0, min(1.0, normalized_confidence))
            
            return lang, normalized_confidence
            
        except Exception as e:
            print(f"    Language detection error: {e}")
            return 'unknown', 0.0
    
    def score_corpus_language(self, corpus_text):
        """Language detection for corpus text"""
        if not corpus_text or len(corpus_text.strip()) < 50:
            return {
                'detected_language': 'unknown',
                'average_confidence': 0.0,
                'language_consistent': False,
                'matches_target': False,
            }
        
        # Sample from different parts
        text_len = len(corpus_text)
        samples = []
        
        if text_len > 0:
            samples.append(corpus_text[:min(500, text_len)])
        if text_len > 1000:
            mid = text_len // 2
            samples.append(corpus_text[mid:min(mid+500, text_len)])
        if text_len > 500:
            samples.append(corpus_text[-500:])
        
        # Detect language for each sample
        detections = []
        for sample in samples:
            if len(sample.strip()) < 10:
                continue
            lang, conf = self.detect_language(sample)
            if lang != 'unknown' and conf > 0:
                detections.append((lang, conf))
                print(f"    Sample detected: {lang} (confidence: {conf:.3f})")
        
        if not detections:
            return {
                'detected_language': 'unknown',
                'average_confidence': 0.0,
                'language_consistent': False,
                'matches_target': False,
            }
        
        # Calculate statistics
        languages = [d[0] for d in detections]
        confidences = [d[1] for d in detections]
        
        most_common_lang = max(set(languages), key=languages.count)
        avg_confidence = sum(confidences) / len(confidences)
        is_consistent = all(lang == most_common_lang for lang in languages)
        
        print(f"    Final language: {most_common_lang}, avg confidence: {avg_confidence:.3f}")
        
        return {
            'detected_language': str(most_common_lang),
            'average_confidence': float(avg_confidence),
            'language_consistent': bool(is_consistent),
            'matches_target': bool(most_common_lang == self.target_language),
        }

In [ ]:
# Cell 8: Progress Manager (adapted for corpus)
class ProgressManager:
    def __init__(self):
        self.progress_data = {
            'processed_chunks': [],
            'current_row': 0,
            'total_chunks_processed': 0,
            'total_qa_pairs_created': 0,
            'request_count': 0,
            'start_time': None,
            'last_update': None,
            'errors': [],
            'statistics': {
                'avg_chunks_per_text': 0.0,
                'avg_qa_per_chunk': 0.0,
                'approach_stats': {'simple': 0, 'deep_thinking': 0, 'iterative_thinking': 0}
            }
        }
        self.load_progress()
    
    def load_progress(self):
        try:
            progress_path = hf_hub_download(
                repo_id=CONFIG['output_repository'],
                filename=CONFIG['progress_file'],
                repo_type="dataset"
            )
            with open(progress_path, 'r') as f:
                saved_progress = json.load(f)
                self.progress_data.update(saved_progress)
            print(f"Progress loaded: {self.progress_data['total_chunks_processed']} chunks processed")
        except Exception as e:
            print(f"No existing progress found, starting fresh: {e}")
            self.progress_data['start_time'] = datetime.now().isoformat()
    
    def save_progress(self):
        try:
            self.progress_data['last_update'] = datetime.now().isoformat()
            
            def convert_types(obj):
                if isinstance(obj, dict):
                    return {k: convert_types(v) for k, v in obj.items()}
                elif isinstance(obj, list):
                    return [convert_types(v) for v in obj]
                elif hasattr(obj, 'item'):
                    return obj.item()
                elif hasattr(obj, 'tolist'):
                    return obj.tolist()
                else:
                    return obj
            
            clean_data = convert_types(self.progress_data)
            local_path = f"./{CONFIG['progress_file']}"
            
            with open(local_path, 'w') as f:
                json.dump(clean_data, f, indent=2)
            
            upload_file(
                path_or_fileobj=local_path,
                path_in_repo=CONFIG['progress_file'],
                repo_id=CONFIG['output_repository'],
                repo_type="dataset",
                commit_message=f"Corpus progress: {self.progress_data['total_qa_pairs_created']} QA pairs"
            )
            print(f"Progress saved to repository")
        except Exception as e:
            print(f"Failed to save progress: {e}")
    
    def is_chunk_processed(self, chunk_id):
        return chunk_id in self.progress_data['processed_chunks']
    
    def mark_chunk_processed(self, chunk_id, qa_count):
        if chunk_id not in self.progress_data['processed_chunks']:
            self.progress_data['processed_chunks'].append(chunk_id)
            self.progress_data['total_chunks_processed'] += 1
            self.progress_data['total_qa_pairs_created'] += qa_count

In [ ]:
# Cell 9: Enhanced Contract Generator - Token Optimized
import random
import math

class ContractCorpusGenerator:
    def __init__(self, progress_manager):
        self.model = genai.GenerativeModel(
            CONFIG['model_name'],
            generation_config=genai.types.GenerationConfig(
                temperature=CONFIG['temperature'],
                max_output_tokens=CONFIG['max_output_tokens'],
                top_p=0.9,
                top_k=40
            ),
            system_instruction=SYSTEM_PROMPT,
            safety_settings=[
                {
                    "category": "HARM_CATEGORY_HARASSMENT",
                    "threshold": "BLOCK_NONE"
                },
                {
                    "category": "HARM_CATEGORY_HATE_SPEECH",
                    "threshold": "BLOCK_NONE"
                },
                {
                    "category": "HARM_CATEGORY_SEXUALLY_EXPLICIT",
                    "threshold": "BLOCK_NONE"
                },
                {
                    "category": "HARM_CATEGORY_DANGEROUS_CONTENT",
                    "threshold": "BLOCK_ONLY_HIGH"
                }
            ]
        )
        
        self.language_scorer = LanguageScorer(DOMAIN_CONFIG['target_language'])
        self.progress_manager = progress_manager
        self.request_count = progress_manager.progress_data['request_count']
    
    def generate_scenario_combination(self):
        """Generate random contract scenario from taxonomy"""
        # Select random number of clauses (3-6 for variety)
        num_clauses = random.randint(3, 6)
        selected_clauses = random.sample(CONTRACT_TAXONOMY['critical_clauses'], k=num_clauses)
        
        scenario = {
            'agreement_type': random.choice(CONTRACT_TAXONOMY['agreement_types']),
            'industry': random.choice(CONTRACT_TAXONOMY['industries']),
            'service_product': random.choice(CONTRACT_TAXONOMY['services'] + CONTRACT_TAXONOMY['products']),
            'relationship': random.choice(CONTRACT_TAXONOMY['relationships']),
            'clauses': ', '.join(selected_clauses),
            'jurisdiction': random.choice(CONTRACT_TAXONOMY['jurisdictions']),
            'risk_profile': random.choice(CONTRACT_TAXONOMY['metadata']['risk_profile']),
            'duration': random.choice(CONTRACT_TAXONOMY['metadata']['duration']),
            'value': random.choice(CONTRACT_TAXONOMY['metadata']['contract_value'])
        }
        return scenario
    
    def generate_contract(self, scenario, corpus_id):
        """Generate realistic contract - single API call"""
        prompt = CONTRACT_GENERATION_PROMPT.format(**scenario)
        
        try:
            time.sleep(6)  # Rate limiting
            response = self.model.generate_content(prompt)
            self.request_count += 1
            self.progress_manager.progress_data['request_count'] = self.request_count
            
            # Handle response
            if not response.candidates:
                print(f"    Contract {corpus_id} - No response")
                return None, None
            
            candidate = response.candidates[0]
            finish_reason = candidate.finish_reason
            
            # Safety filter hit
            if finish_reason == 2:
                print(f"    Contract {corpus_id} - Safety filtered, retrying with safer approach")
                return self.generate_safe_fallback(scenario, corpus_id)
            
            # Get contract text
            contract_text = candidate.content.parts[0].text.strip()
            
            # Handle truncation
            if finish_reason == 3:
                print(f"    Contract {corpus_id} - Reached max tokens ({len(contract_text)} chars)")
                # Contracts can be long - this is OK, just note it
                # If you want continuation, uncomment:
                # continuation = self.continue_contract(contract_text, scenario)
                # if continuation:
                #     contract_text += "\n\n" + continuation
            
            print(f"    Generated: {len(contract_text)} chars, {len(contract_text.split())} words")
            
            # Quality check
            if len(contract_text) < CONFIG['min_contract_length']:
                print(f"    Contract too short, skipping")
                return None, None
            
            # Generate lightweight fairness data
            fairness_data = self.analyze_contract_quick(contract_text, scenario)
            
            return contract_text, fairness_data
            
        except Exception as e:
            print(f"    Error: {e}")
            return None, None
    
    def generate_safe_fallback(self, scenario, corpus_id):
        """Fallback with extra-safe language when filtered"""
        safe_prompt = f"""Draft a standard commercial {scenario['agreement_type']} for the {scenario['industry']} industry.

Context:
- Business purpose: {scenario['service_product']}
- Contract term: {scenario['duration']}
- Approximate value: {scenario['value']}
- Applicable law: {scenario['jurisdiction']}

Create a complete, professional agreement with:
- Identification of parties
- Scope and deliverables
- Commercial terms
- Standard protective provisions
- Term and termination clauses
- General provisions
- Execution section

Use standard business language. Be specific with terms and conditions.

AGREEMENT:"""
        
        try:
            time.sleep(6)
            response = self.model.generate_content(safe_prompt)
            self.request_count += 1
            
            if not response.candidates or response.candidates[0].finish_reason == 2:
                print(f"    Fallback also filtered - skipping")
                return None, None
            
            contract_text = response.candidates[0].content.parts[0].text.strip()
            
            if len(contract_text) < CONFIG['min_contract_length']:
                return None, None
            
            fairness_data = self.analyze_contract_quick(contract_text, scenario)
            return contract_text, fairness_data
            
        except Exception as e:
            print(f"    Fallback failed: {e}")
            return None, None
    
    def analyze_contract_quick(self, contract_text, scenario):
        """Quick structural analysis - NO extra API call unless enabled"""
        
        # Count major sections
        section_count = len(re.findall(r'(?i)^(?:article|section|clause|\d+\.)\s+', contract_text, re.MULTILINE))
        word_count = len(contract_text.split())
        
        # Map relationship to party roles
        role_map = {
            'B2B (Business-to-Business)': ('Service Provider', 'Client'),
            'B2G (Business-to-Government)': ('Contractor', 'Government Entity'),
            'Employer-Employee': ('Employer', 'Employee'),
            'Partner-Partner (Joint Venture)': ('Partner A', 'Partner B'),
            'Licensor-Licensee (IP Transfer)': ('Licensor', 'Licensee'),
            'Vendor-Client': ('Vendor', 'Client'),
            'Franchisor-Franchisee': ('Franchisor', 'Franchisee'),
        }
        
        party_roles = role_map.get(scenario['relationship'], ('Party A', 'Party B'))
        
        # Heuristic fairness score based on agreement type
        fairness_map = {
            'Employment Agreement': 4,  # Usually favors employer
            'Franchise Agreement': 4,  # Favors franchisor
            'Software License Agreement (EULA)': 3,  # Strongly favors licensor
            'Partnership Agreement': 5,  # Generally balanced
            'Joint Venture Agreement': 5,
            'Non-Disclosure Agreement (NDA)': 5,
            'Service Level Agreement (SLA)': 4,  # Slight vendor advantage
        }
        
        fairness_score = fairness_map.get(scenario['agreement_type'], 5)
        
        if fairness_score <= 3:
            balance_rating = "Strongly Favors Party A"
        elif fairness_score == 4:
            balance_rating = "Slightly Favors Party A"
        elif fairness_score == 5:
            balance_rating = "Balanced"
        elif fairness_score == 6:
            balance_rating = "Slightly Favors Party B"
        else:
            balance_rating = "Strongly Favors Party B"
        
        # Optional: Detailed analysis (costs extra API call)
        if CONFIG.get('enable_fairness_analysis', False):
            detailed_analysis = self.get_detailed_fairness(contract_text[:2000], scenario, party_roles)
        else:
            detailed_analysis = f"Standard {scenario['agreement_type']} with typical provisions for {scenario['industry']} sector. Contains {section_count} major sections covering commercial terms, obligations, and protective clauses appropriate for {scenario['jurisdiction']}."
        
        return {
            'fairness_score': fairness_score,
            'balance_rating': balance_rating,
            'party_a_role': party_roles[0],
            'party_b_role': party_roles[1],
            'party_a_advantages': 0,
            'party_b_advantages': 0,
            'balanced_clauses': section_count,
            'total_analyzed_clauses': section_count,
            'analysis_text': detailed_analysis
        }
    
    def get_detailed_fairness(self, contract_preview, scenario, party_roles):
        """Optional detailed analysis - only if enabled"""
        prompt = FAIRNESS_ANALYSIS_PROMPT.format(
            contract_preview=contract_preview,
            agreement_type=scenario['agreement_type'],
            industry=scenario['industry'],
            relationship=scenario['relationship']
        )
        
        try:
            time.sleep(4)
            response = self.model.generate_content(prompt)
            self.request_count += 1
            
            if response.candidates and response.candidates[0].finish_reason not in [2, 4]:
                return response.candidates[0].content.parts[0].text.strip()
        except:
            pass
        
        return f"Analysis unavailable - standard {scenario['agreement_type']} terms apply."
    
    def generate_batch(self, batch_size):
        """Generate batch of contracts"""
        results = []
        
        for i in range(batch_size):
            print(f"\nGenerating contract {i+1}/{batch_size}")
            
            scenario = self.generate_scenario_combination()
            print(f"  {scenario['agreement_type']} | {scenario['industry']}")
            print(f"  Value: {scenario['value']} | Term: {scenario['duration']}")
            
            contract_text, fairness_data = self.generate_contract(scenario, i)
            
            if not contract_text or not fairness_data:
                continue
            
            # Language validation
            language_scores = self.language_scorer.score_corpus_language(contract_text)
            
            result = {
                'corpus_id': f"contract_{self.progress_manager.progress_data['total_qa_pairs_created'] + len(results):05d}",
                'contract_text': contract_text,
                'text_length': len(contract_text),
                'word_count': len(contract_text.split()),
                
                # Scenario metadata
                'agreement_type': scenario['agreement_type'],
                'industry': scenario['industry'],
                'service_product': scenario['service_product'],
                'party_relationship': scenario['relationship'],
                'key_clauses': scenario['clauses'],
                'jurisdiction': scenario['jurisdiction'],
                'risk_profile': scenario['risk_profile'],
                'duration': scenario['duration'],
                'contract_value': scenario['value'],
                
                # Fairness metrics
                'fairness_score': fairness_data['fairness_score'],
                'balance_rating': fairness_data['balance_rating'],
                'party_a_role': fairness_data['party_a_role'],
                'party_b_role': fairness_data['party_b_role'],
                'party_a_advantages': fairness_data['party_a_advantages'],
                'party_b_advantages': fairness_data['party_b_advantages'],
                'balanced_clauses': fairness_data['balanced_clauses'],
                'total_analyzed_clauses': fairness_data['total_analyzed_clauses'],
                'fairness_analysis': fairness_data['analysis_text'],
                
                # Language
                **language_scores,
                
                'timestamp': datetime.now().isoformat(),
            }
            
            results.append(result)
            print(f"  ✓ Complete: {result['word_count']} words | Fairness: {result['fairness_score']}/10 ({result['balance_rating']})")
        
        return results

In [ ]:
# Cell 10: Contract Corpus Processing Functions
def process_contract_corpus_batch(target_count):
    """Generate a batch of contract corpus texts"""
    generator = ContractCorpusGenerator(progress_manager)
    
    already_generated = progress_manager.progress_data.get('total_qa_pairs_created', 0)
    remaining = target_count - already_generated
    
    if remaining <= 0:
        print(f"Target of {target_count} corpus texts already reached")
        return
    
    batch_size = min(remaining, CONFIG['batch_size'])
    
    print(f"Generating {batch_size} contract corpus texts ({already_generated}/{target_count} complete)")
    
    results = generator.generate_batch(batch_size)
    
    if results:
        save_corpus_batch(results, already_generated)
        
        # Update progress
        progress_manager.progress_data['total_qa_pairs_created'] += len(results)
        progress_manager.save_progress()
        
        print(f"\nBatch complete: {len(results)} corpus texts generated")
        print(f"Total progress: {progress_manager.progress_data['total_qa_pairs_created']}/{target_count}")

def save_corpus_batch(results, batch_start):
    """Save corpus batch with standardized format"""
    if not results:
        return
    
    try:
        df = pd.DataFrame(results)
        
        # Standardize schema
        df = df.astype({
            'corpus_id': 'string',
            'contract_text': 'string',
            'agreement_type': 'string',
            'industry': 'string',
            'service_product': 'string',
            'party_relationship': 'string',
            'key_clauses': 'string',
            'jurisdiction': 'string',
            'risk_profile': 'string',
            'duration': 'string',
            'contract_value': 'string',
        })
        
        batch_id = f"{batch_start:05d}-{batch_start+len(results):05d}"
        filename = f"train-{batch_id}.parquet"
        
        temp_filepath = f"/tmp/{filename}"
        df.to_parquet(temp_filepath, index=False, engine='pyarrow')
        
        upload_file(
            path_or_fileobj=temp_filepath,
            path_in_repo=filename,
            repo_id=CONFIG['output_repository'],
            repo_type="dataset",
            commit_message=f"Contract corpus batch {batch_id}: {len(results)} texts"
        )
        
        print(f"  ✓ Uploaded {filename} ({len(results)} corpus texts)")
        
        os.remove(temp_filepath)
        
    except Exception as e:
        print(f"Failed to save corpus batch: {e}")

def continue_contract_generation(target_count=100):
    """Continue generating contract corpus until target reached"""
    while progress_manager.progress_data.get('total_qa_pairs_created', 0) < target_count:
        process_contract_corpus_batch(target_count)

# Cell 11 additions
def show_contract_stats():
    """Show contract generation statistics"""
    print(f"Contract Corpus Generation Stats:")
    print(f"  Total corpus texts: {progress_manager.progress_data.get('total_qa_pairs_created', 0)}")
    print(f"  API requests: {progress_manager.progress_data.get('request_count', 0)}")

In [ ]:
# Cell 11: Utility Functions
def show_corpus_config():
    """Display current corpus configuration"""
    print("Corpus Synthesis Configuration:")
    print(f"  Domain: {DOMAIN_CONFIG['domain']}")
    print(f"  Approach: {DOMAIN_CONFIG['approach']}")
    print(f"  Source Dataset: {CONFIG['source_dataset']}")
    print(f"  Corpus Column: {CONFIG['corpus_column']}")
    print(f"  Variants per text: {DOMAIN_CONFIG['num_variants']}")
    print(f"  Max text length: {DOMAIN_CONFIG['max_text_length']}")

def update_corpus_config(corpus_column=None, approach=None, num_variants=None, max_text_length=None):
    """Update corpus configuration"""
    if corpus_column:
        CONFIG['corpus_column'] = corpus_column
    if approach:
        DOMAIN_CONFIG['approach'] = approach
    if num_variants:
        DOMAIN_CONFIG['num_variants'] = num_variants
    if max_text_length:
        DOMAIN_CONFIG['max_text_length'] = max_text_length
    
    print("Updated corpus configuration")
    show_corpus_config()

def show_progress():
    """Show current progress with enhanced file validation"""
    stats = progress_manager.progress_data
    print(f"Corpus Synthesis Progress:")
    print(f"  Current row: {stats.get('current_row', 0)}")
    print(f"  Processed rows: {len(stats.get('processed_rows', []))}")
    print(f"  QA pairs created: {stats['total_qa_pairs_created']}")
    print(f"  API requests: {stats['request_count']}")
    
    # Validate HF dataset compatibility
    validate_dataset_files()

# Keep the rest of Cell 11 as is...

## RUN

In [ ]:
# Initialize progress manager
progress_manager = ProgressManager()

print("Corpus-Based QA Synthesis Pipeline Ready!")
print("=" * 50)
print("Key Functions:")
print("- show_corpus_config() - Display current configuration")
print("- update_corpus_config(corpus_column, chunk_method, chunk_size, approach) - Update settings")
print("- continue_corpus_synthesis() - Start/continue synthesis")
print("- process_corpus_batch(start_row, end_row) - Process specific rows")
print("- show_progress() - Check current progress")
print("\nFirst, update CONFIG with your dataset info, then run: continue_corpus_synthesis()")

#show_corpus_config()
#continue_corpus_synthesis()
# Generate 100 contract corpus texts
continue_contract_generation(target_count=1000000)

## END